# Training klasifikasi gender (PA-100K) dengan YOLO11-cls

**Jalankan di Google Colab**, bukan di Mac. Runtime → Change runtime type → **T4 GPU**.

Runtime → **Run all**. Jika terputus, Run all lagi: training lanjut otomatis dari checkpoint di Drive.

Hasil: `MyDrive/cctv_resto/models/gender_classifier.pt`

> Gender di sistem ini hanya **estimasi statistik agregat** (voting beberapa frame per orang), akurasi terbatas dan bisa bias. Jangan dipakai menilai individu.

In [ ]:
# 1) Cek GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "GPU tidak aktif: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2) Install dependensi
!pip -q install ultralytics

In [ ]:
# 3) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os, random, shutil, zipfile
from collections import Counter
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/cctv_resto')
MODELS, RUNS = ROOT / 'models', ROOT / 'runs'
for p in (MODELS, RUNS):
    p.mkdir(parents=True, exist_ok=True)
RUN_NAME = 'gender'
RAW, DS = Path('/content/pa100k_raw'), Path('/content/gender_ds')

In [ ]:
# 4) Download dataset: PA-100K gender split (CC0, 89.7 MB, gambar 75x75, ~49 ribu gambar)
# Sumber utama: 4TU.ResearchData (mirror resmi). Cadangan: Figshare (sempat error HTTP 504).
# Jika keduanya gagal: download manual lewat browser, upload ke MyDrive/cctv_resto/datasets/pa100k_gender.zip
import hashlib, subprocess, time
URLS = [
    'https://data.4tu.nl/file/a6c8b6dd-0df6-4de8-806b-e745074d10a8/f2dc979f-b696-4781-aec3-2827c5f9f836',
    'https://ndownloader.figshare.com/files/24152912',
]
MD5 = 'cc2f2550d03bd74b77ba8e9942421c81'  # checksum resmi: memastikan file utuh
ZIP = Path('/content/pa100k_gender.zip')
DRIVE_ZIP = ROOT / 'datasets' / 'pa100k_gender.zip'

def valid(path):
    return path.exists() and hashlib.md5(path.read_bytes()).hexdigest() == MD5

if not valid(ZIP) and DRIVE_ZIP.exists():
    shutil.copy(DRIVE_ZIP, ZIP)
for attempt in range(6):
    if valid(ZIP):
        break
    url = URLS[attempt % len(URLS)]
    print(f'download percobaan {attempt + 1}/6 dari {url.split("/")[2]} ...')
    subprocess.run(['wget', '-q', '-O', str(ZIP), url])
    if not valid(ZIP):
        time.sleep(10)
if not valid(ZIP):
    raise SystemExit('Download gagal. Upload zip manual ke ' + str(DRIVE_ZIP))
if not RAW.exists():
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall(RAW)
print('dataset siap diekstrak, MD5 cocok')

In [ ]:
# 5) Temukan gambar + label dari struktur folder, lalu susun train/val/test
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}
LABELS = {'male': 'male', 'man': 'male', 'female': 'female', 'woman': 'female'}  # nama folder -> label
SPLITS = {'train': 'train', 'training': 'train', 'val': 'val', 'valid': 'val', 'validation': 'val', 'test': 'test'}

items = []  # (path, label, split atau None)
for p in RAW.rglob('*'):
    if p.suffix.lower() not in IMG_EXT:
        continue
    parts = [x.lower() for x in p.relative_to(RAW).parts[:-1]]
    label = next((LABELS[x] for x in reversed(parts) if x in LABELS), None)
    split = next((SPLITS[x] for x in parts if x in SPLITS), None)
    if label:
        items.append((p, label, split))

if not items:  # struktur zip tidak seperti dugaan: tampilkan agar nama folder bisa dicocokkan di LABELS
    for d in sorted(RAW.rglob('*')):
        if d.is_dir() and len(d.relative_to(RAW).parts) <= 3:
            print(d.relative_to(RAW))
    raise SystemExit('Tidak ada gambar berlabel male/female. Sesuaikan LABELS dengan nama folder di atas.')

print('struktur asli:', Counter((s, l) for _, l, s in items))
if {s for _, _, s in items} != {'train', 'val', 'test'}:  # split bawaan tidak lengkap -> acak 80/10/10
    random.seed(0)
    random.shuffle(items)
    n = len(items)
    items = [(p, l, 'train' if i < 0.8 * n else 'val' if i < 0.9 * n else 'test') for i, (p, l, _) in enumerate(items)]
    print('split bawaan tidak lengkap, dibuat ulang 80/10/10')

shutil.rmtree(DS, ignore_errors=True)
for i, (p, label, split) in enumerate(items):
    dst = DS / split / label
    dst.mkdir(parents=True, exist_ok=True)
    os.symlink(p, dst / f'{i}_{p.name}')  # symlink: cepat, tanpa menggandakan 100k file
print('dataset siap:', Counter((s, l) for _, l, s in items))

In [ ]:
# 6) Training. Bisa dilanjutkan jika Colab terputus.
# Hyperparameter:
#   model  yolo11n-cls.pt -> kecil & cepat; cukup untuk 2 class
#   imgsz 96   -> gambar sumber 75x75, jadi 96 sudah cukup (bawaan 224 hanya membuang waktu)
#   epochs 30, patience 8 -> berhenti jika akurasi val tidak naik 8 epoch
#   batch 128  -> aman di T4 untuk gambar sekecil ini
from ultralytics import YOLO

last = RUNS / RUN_NAME / 'weights' / 'last.pt'

def finished(path):
    return torch.load(path, map_location='cpu', weights_only=False).get('optimizer') is None

if last.exists() and not finished(last):
    print('Checkpoint ditemukan, melanjutkan training...')
    YOLO(str(last)).train(resume=True)
elif last.exists():
    print('Training sebelumnya sudah selesai, lanjut ke evaluasi.')
else:
    YOLO('yolo11n-cls.pt').train(
        data=str(DS), epochs=30, imgsz=96, batch=128, patience=8, workers=2,
        device=0, project=str(RUNS), name=RUN_NAME, exist_ok=True, seed=0,
    )

In [ ]:
# 7) Evaluasi di data test: accuracy, F1, confusion matrix
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt

best = RUNS / RUN_NAME / 'weights' / 'best.pt'
model = YOLO(str(best))
files = sorted(p for p in (DS / 'test').rglob('*') if p.suffix.lower() in IMG_EXT)
y_true = [p.parent.name for p in files]
y_pred = [model.names[r.probs.top1] for r in model.predict([str(f) for f in files], imgsz=96, stream=True, verbose=False)]

labels = sorted(set(y_true))
print(f"accuracy : {accuracy_score(y_true, y_pred):.3f}")
print(f"F1 macro : {f1_score(y_true, y_pred, average='macro'):.3f}")
print(classification_report(y_true, y_pred, labels=labels, digits=3))
cm = confusion_matrix(y_true, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels)), labels); ax.set_yticks(range(len(labels)), labels)
ax.set_xlabel('prediksi'); ax.set_ylabel('sebenarnya')
for (i, j), v in __import__('numpy').ndenumerate(cm):
    ax.text(j, i, v, ha='center', va='center')
plt.show()

In [ ]:
# 8) Simpan ke Drive + export ONNX
shutil.copy(best, MODELS / 'gender_classifier.pt')
onnx = model.export(format='onnx', imgsz=96)
shutil.copy(onnx, MODELS / 'gender_classifier.onnx')
(MODELS / 'GENDER_DATASET.md').write_text(
    "PA-100K gender split (4TU.ResearchData / Figshare 12763712), lisensi CC0. Sumber asli: Liu dkk., HydraPlus-Net.\n")

print("""
SELESAI. Cara pakai di Mac:
1. Buka drive.google.com > My Drive > cctv_resto > models
2. Download gender_classifier.pt
3. Taruh di folder project:  cctv/models/gender_classifier.pt
4. Jalankan seperti biasa. Estimasi gender aktif otomatis jika file itu ada.
   Label 'unknown' dipakai jika confidence rendah atau voting kurang dari 5 frame.
""")
print(sorted(p.name for p in MODELS.iterdir()))